# Final Testing — Swift v2.0 vs PyBullet

## Upload these 3 files when Cell 1 runs

| File to upload | Version |
|---|---|
| `swift_physics_headless_wind_latest.py` | v2.0 (blade flapping + hub drag + ground effect) |
| `swift_physics_headless_rlvec_latest.py` | v2.0 |
| `swift_rl_env_latest.py` | current |

**Cell 2 installs PyBullet. All other cells compare Swift v2.0 vs v1.1 vs PyBullet.**

In [ ]:
# ── Cell 1: Upload latest files ───────────────────────────────────────────────
from google.colab import files
import os, sys, glob

print('Upload: swift_physics_headless_wind_latest.py')
print('        swift_physics_headless_rlvec_latest.py')
print('        swift_rl_env_latest.py')
uploaded = files.upload()

renames = {
    'swift_physics_headless_wind_latest.py':  'swift_physics_headless_wind.py',
    'swift_physics_headless_rlvec_latest.py': 'swift_physics_headless_rlvec.py',
    'swift_rl_env_latest.py':                 'swift_rl_env.py',
}
for src, dst in renames.items():
    sp, dp = f'/content/{src}', f'/content/{dst}'
    if os.path.exists(sp):
        os.replace(sp, dp); print(f'  OK  {src} -> {dst}')
    else:
        print(f'  !!  {src} not found -- upload it!')

for mod in ['swift_physics_headless_wind', 'swift_physics_headless_rlvec', 'swift_rl_env']:
    if mod in sys.modules: del sys.modules[mod]
    for pyc in glob.glob(f'/content/__pycache__/{mod}*.pyc'): os.remove(pyc)

print('\nDone. Run Cell 2 next.')

In [ ]:
# ── Cell 2: Install PyBullet + imports ────────────────────────────────────────
!pip install pybullet -q

import numpy as np
import matplotlib.pyplot as plt
import pybullet as pb
import pybullet_data

import swift_physics_headless_wind  as wind
import swift_physics_headless_rlvec as rlvec

assert '2.0' in wind.__version__,  f'Need v2.0 wind, got {wind.__version__}'
assert '2.0' in rlvec.__version__, f'Need v2.0 rlvec, got {rlvec.__version__}'

print(f'wind  : {wind.__version__}')
print(f'rlvec : {rlvec.__version__}')
print(f'pybullet : {pb.__version__}')
print('All imports OK.')

In [ ]:
# ── Cell 3: PyBullet + Swift helpers ──────────────────────────────────────────
params    = wind.PARAMS_NANOBENCH
params_v11 = {**params, 'C_FLAP': 0.0, 'C_HUB': 0.0, 'R_ROTOR': 0.0}
K_DRAG    = 0.02
DT        = 0.01

# ── PyBullet setup ────────────────────────────────────────────────────────────
def setup_pybullet(p0, q0_wxyz, dt=0.01):
    """Create a fresh headless PyBullet simulation with a Crazyflie-mass body."""
    cid = pb.connect(pb.DIRECT)
    pb.setGravity(0, 0, -9.81, physicsClientId=cid)
    pb.setTimeStep(dt, physicsClientId=cid)
    pb.setAdditionalSearchPath(pybullet_data.getDataPath(), physicsClientId=cid)

    m = params['m']
    col_id = pb.createCollisionShape(pb.GEOM_BOX, halfExtents=[0.04, 0.04, 0.01],
                                      physicsClientId=cid)
    vis_id = pb.createVisualShape(pb.GEOM_BOX, halfExtents=[0.04, 0.04, 0.01],
                                   physicsClientId=cid)
    # PyBullet quaternion convention: [x, y, z, w]
    w, x, y, z = q0_wxyz
    body_id = pb.createMultiBody(
        baseMass=m,
        baseCollisionShapeIndex=col_id,
        baseVisualShapeIndex=vis_id,
        basePosition=p0.tolist(),
        baseOrientation=[x, y, z, w],
        physicsClientId=cid)
    J_diag = np.diag(params['J']).tolist()
    pb.changeDynamics(body_id, -1, localInertiaDiagonal=J_diag,
                       physicsClientId=cid)
    return cid, body_id


def pybullet_step(cid, body_id, Omega, wind_vel=None):
    """Apply thrust + drag torques to PyBullet body and step simulation."""
    pos, orn = pb.getBasePositionAndOrientation(body_id, physicsClientId=cid)
    lin_vel, _ = pb.getBaseVelocity(body_id, physicsClientId=cid)
    R = np.array(pb.getMatrixFromQuaternion(orn)).reshape(3, 3)
    c_l, c_d = params['c_l'], params['c_d']
    r_P, ss  = params['r_P'], params['spin_sign']

    for j in range(4):
        thrust  = c_l * Omega[j]**2
        f_world = (R @ np.array([0., 0., thrust])).tolist()
        p_world = (np.array(pos) + R @ r_P[j]).tolist()
        pb.applyExternalForce(body_id, -1, f_world, p_world,
                               pb.WORLD_FRAME, physicsClientId=cid)
        tau_z   = ss[j] * c_d * Omega[j]**2
        tau_w   = (R @ np.array([0., 0., tau_z])).tolist()
        pb.applyExternalTorque(body_id, -1, tau_w, pb.WORLD_FRAME,
                                physicsClientId=cid)

    # Wind drag (same K_DRAG model as Swift)
    if wind_vel is not None:
        F_wind = K_DRAG * (np.array(wind_vel) - np.array(lin_vel))
        pb.applyExternalForce(body_id, -1, F_wind.tolist(), pos,
                               pb.WORLD_FRAME, physicsClientId=cid)

    pb.stepSimulation(physicsClientId=cid)


def pybullet_read(cid, body_id):
    """Read [p, q_wxyz, v, omega] from PyBullet."""
    pos, orn  = pb.getBasePositionAndOrientation(body_id, physicsClientId=cid)
    lin, ang  = pb.getBaseVelocity(body_id, physicsClientId=cid)
    x, y, z, w = orn
    return (np.array(pos), np.array([w, x, y, z]),
            np.array(lin), np.array(ang))


# ── Swift runner ──────────────────────────────────────────────────────────────
def make_state(p0, v0=None, q0=None):
    return dict(
        p_WB    = np.array(p0, dtype=float),
        q_WB    = np.array(q0 or [1., 0., 0., 0.]),
        v_WB    = np.array(v0 or [0., 0., 0.], dtype=float),
        omega_B = np.zeros(3),
        Omega   = wind.hover_omega_from_params(params).copy(),
    )


def run_swift(p_params, p0, v0=None, wind_fn=None, T_END=6.0):
    state = make_state(p0, v0)
    Om    = wind.hover_omega_from_params(p_params)
    N     = int(T_END / DT)
    traj  = {'p':[], 'v':[], 'q':[], 'ge':[], 't':[]}
    for k in range(N):
        t = k * DT
        f_ps, t_ps = wind.propeller_force_torque(Om, p_params['c_l'], p_params['c_d'], p_params['spin_sign'])
        f_b, tau_b = wind.aggregate_forces(f_ps, t_ps, p_params['r_P'])
        g_eff = p_params['g_W'].copy()
        if wind_fn:
            F_w   = K_DRAG * (wind_fn(t) - state['v_WB'])
            g_eff = p_params['g_W'] + F_w / p_params['m']
        state['Omega'] = Om
        state = wind.rk4_rigid_body_step(state, f_b, tau_b, {**p_params, 'g_W': g_eff}, DT)
        ge = wind.ground_effect_factor(float(state['p_WB'][2]), p_params.get('R_ROTOR', 0.023))
        traj['p'].append(state['p_WB'].copy()); traj['v'].append(state['v_WB'].copy())
        traj['q'].append(state['q_WB'].copy()); traj['ge'].append(ge); traj['t'].append(t)
    for k in ('p','v','q','ge','t'): traj[k] = np.array(traj[k])
    return traj


def run_pybullet(p0, v0=None, wind_fn=None, T_END=6.0):
    Om   = wind.hover_omega_from_params(params)
    q0   = np.array([1., 0., 0., 0.])
    cid, bid = setup_pybullet(np.array(p0), q0, dt=DT)
    if v0 is not None:
        pb.resetBaseVelocity(bid, linearVelocity=list(v0), physicsClientId=cid)
    N    = int(T_END / DT)
    traj = {'p':[], 'v':[], 'q':[], 't':[]}
    for k in range(N):
        t = k * DT
        w_vel = wind_fn(t) if wind_fn else None
        pybullet_step(cid, bid, Om, wind_vel=w_vel)
        p, q, v, _ = pybullet_read(cid, bid)
        traj['p'].append(p); traj['v'].append(v)
        traj['q'].append(q); traj['t'].append(t)
    pb.disconnect(cid)
    for k in ('p','v','q','t'): traj[k] = np.array(traj[k])
    return traj


def tilt_deg(traj):
    return np.degrees([2*np.arctan2(np.linalg.norm(q[1:3]), q[0]) for q in traj['q']])


print('Helpers ready.')
print(f'Hover Omega : {wind.hover_omega_from_params(params)[0]:.1f} rad/s')
print(f'GE active below: {4*params["R_ROTOR"]*100:.1f} cm')

In [ ]:
# ── Cell 4: Standard 6s Trajectory (Swift v2.0 vs v1.1 vs PyBullet) ──────────
# Same test as the original comparison notebook but now using v2.0 physics
T_TRAJ = 6.0
wind_fn_std = lambda t: np.array([2.5*np.sin(t), -1.5*np.sin(0.7*t+0.5), 0.3*np.cos(1.3*t)])

print('Running standard 6s trajectory comparison...')
traj_v2   = run_swift(params,     p0=[0.,0.,1.], wind_fn=wind_fn_std, T_END=T_TRAJ)
traj_v11  = run_swift(params_v11, p0=[0.,0.,1.], wind_fn=wind_fn_std, T_END=T_TRAJ)
traj_pb   = run_pybullet(p0=[0.,0.,1.], wind_fn=wind_fn_std, T_END=T_TRAJ)
print('Done!')

# Position deviation from PyBullet (ground truth)
dev_v2  = np.linalg.norm(traj_v2['p']  - traj_pb['p'], axis=1) * 1000  # mm
dev_v11 = np.linalg.norm(traj_v11['p'] - traj_pb['p'], axis=1) * 1000  # mm

print(f'\n=== Deviation from PyBullet ===')
print(f'Swift v2.0 : max = {dev_v2.max():.2f} mm  |  final = {dev_v2[-1]:.2f} mm')
print(f'Swift v1.1 : max = {dev_v11.max():.2f} mm  |  final = {dev_v11[-1]:.2f} mm')
change = (dev_v2.max() - dev_v11.max()) / dev_v11.max() * 100
direction = 'CLOSER' if change < 0 else 'further'
print(f'v2.0 is {abs(change):.1f}% {direction} to PyBullet than v1.1')

In [ ]:
# ── Cell 5: Extreme Cases (Swift v2.0 vs v1.1 vs PyBullet) ───────────────────
T_EXT = 3.0
print('Running extreme scenarios...')

# S1: Near-ground (5 cm) -- ground effect
s1_v2  = run_swift(params,     p0=[0.,0.,0.05], T_END=T_EXT)
s1_v11 = run_swift(params_v11, p0=[0.,0.,0.05], T_END=T_EXT)
s1_pb  = run_pybullet(         p0=[0.,0.,0.05], T_END=T_EXT)
print('  S1 done (near-ground 5 cm)')

# S2: High-speed (8 m/s forward) -- hub drag + blade flapping
s2_v2  = run_swift(params,     p0=[0.,0.,1.], v0=[8.,0.,0.], T_END=T_EXT)
s2_v11 = run_swift(params_v11, p0=[0.,0.,1.], v0=[8.,0.,0.], T_END=T_EXT)
s2_pb  = run_pybullet(         p0=[0.,0.,1.], v0=[8.,0.,0.], T_END=T_EXT)
print('  S2 done (high-speed 8 m/s)')

# S3: Extreme wind (10 m/s crosswind)
w10 = lambda t: np.array([10., 0., 0.])
s3_v2  = run_swift(params,     p0=[0.,0.,1.], wind_fn=w10, T_END=T_EXT)
s3_v11 = run_swift(params_v11, p0=[0.,0.,1.], wind_fn=w10, T_END=T_EXT)
s3_pb  = run_pybullet(         p0=[0.,0.,1.], wind_fn=w10, T_END=T_EXT)
print('  S3 done (10 m/s crosswind)')

# S4: Very close ground (2 cm) -- maximum ground effect
s4_v2  = run_swift(params,     p0=[0.,0.,0.02], T_END=T_EXT)
s4_v11 = run_swift(params_v11, p0=[0.,0.,0.02], T_END=T_EXT)
s4_pb  = run_pybullet(         p0=[0.,0.,0.02], T_END=T_EXT)
print('  S4 done (very close ground 2 cm)')

print('\nAll extreme scenarios complete!')

In [ ]:
# ── Cell 6: Plots ─────────────────────────────────────────────────────────────
C2, C11, CPB = '#e74c3c', '#95a5a6', '#2c3e50'

def trio(ax, t, y2, y11, ypb, title, ylabel, hline=None, hlab=''):
    ax.plot(t, ypb, color=CPB, lw=2,   label='PyBullet', zorder=3)
    ax.plot(t, y2,  color=C2,  lw=1.8, ls='-',  label='Swift v2.0', zorder=2)
    ax.plot(t, y11, color=C11, lw=1.5, ls='--', label='Swift v1.1', zorder=1)
    if hline is not None: ax.axhline(hline, color='gray', ls=':', lw=1, label=hlab)
    ax.set_title(title, fontsize=9); ax.set_xlabel('t (s)', fontsize=8)
    ax.set_ylabel(ylabel, fontsize=8); ax.legend(fontsize=7); ax.grid(alpha=0.3)

# ── Figure 1: Standard 6s trajectory ─────────────────────────────────────────
fig1, axes1 = plt.subplots(1, 3, figsize=(20, 5))
fig1.suptitle('Standard 6s Trajectory with Wind  |  Swift v2.0 vs v1.1 vs PyBullet',
              fontweight='bold')
trio(axes1[0], traj_v2['t'], traj_v2['p'][:,2], traj_v11['p'][:,2], traj_pb['p'][:,2],
     'Altitude', 'z (m)')
trio(axes1[1], traj_v2['t'], dev_v2, dev_v11, np.zeros(len(dev_v2)),
     'Position Deviation from PyBullet', 'error (mm)')
trio(axes1[2], traj_v2['t'], tilt_deg(traj_v2), tilt_deg(traj_v11), tilt_deg(traj_pb),
     'Tilt Angle', 'tilt (deg)')
fig1.tight_layout()
fig1.savefig('traj_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Extreme cases ───────────────────────────────────────────────────
fig2, axes2 = plt.subplots(4, 3, figsize=(21, 18))
fig2.suptitle('Extreme Cases  |  BLACK=PyBullet  RED=Swift v2.0  GREY=Swift v1.1',
              fontweight='bold', fontsize=12)

# S1
axes2[0,0].plot(s1_v2['t'], s1_v2['ge'],  color=C2,  lw=2,   label='v2.0 GE')
axes2[0,0].plot(s1_v11['t'],s1_v11['ge'], color=C11, lw=1.5, ls='--', label='v1.1 (no GE)')
axes2[0,0].axhline(1.0, color='gray', ls=':', label='k_ge=1'); axes2[0,0].legend(fontsize=7)
axes2[0,0].set_title('S1: Ground Effect Factor  (start z=5cm)'); axes2[0,0].set_ylabel('k_ge')
trio(axes2[0,1], s1_v2['t'], s1_v2['p'][:,2]*100, s1_v11['p'][:,2]*100, s1_pb['p'][:,2]*100,
     'S1: Altitude (cm)', 'z (cm)', hline=9.2, hlab='4×R_rotor')
trio(axes2[0,2], s1_v2['t'], s1_v2['v'][:,2], s1_v11['v'][:,2], s1_pb['v'][:,2],
     'S1: Vertical Velocity', 'vz (m/s)')

# S2
trio(axes2[1,0], s2_v2['t'], s2_v2['v'][:,0], s2_v11['v'][:,0], s2_pb['v'][:,0],
     'S2: Forward Velocity  (hub drag)', 'vx (m/s)')
trio(axes2[1,1], s2_v2['t'], tilt_deg(s2_v2), tilt_deg(s2_v11), tilt_deg(s2_pb),
     'S2: Tilt  (blade flapping)', 'tilt (deg)')
trio(axes2[1,2], s2_v2['t'], s2_v2['p'][:,2], s2_v11['p'][:,2], s2_pb['p'][:,2],
     'S2: Altitude During High-Speed', 'z (m)')

# S3
trio(axes2[2,0], s3_v2['t'], s3_v2['p'][:,0], s3_v11['p'][:,0], s3_pb['p'][:,0],
     'S3: X Drift  (10 m/s crosswind)', 'x (m)')
trio(axes2[2,1], s3_v2['t'], tilt_deg(s3_v2), tilt_deg(s3_v11), tilt_deg(s3_pb),
     'S3: Tilt Under 10 m/s Wind', 'tilt (deg)')
trio(axes2[2,2], s3_v2['t'], s3_v2['v'][:,0], s3_v11['v'][:,0], s3_pb['v'][:,0],
     'S3: X Velocity (hub drag vs wind)', 'vx (m/s)')

# S4
axes2[3,0].plot(s4_v2['t'], s4_v2['ge'],  color=C2,  lw=2,   label='v2.0')
axes2[3,0].plot(s4_v11['t'],s4_v11['ge'], color=C11, lw=1.5, ls='--', label='v1.1')
axes2[3,0].axhline(1.0, color='gray', ls=':'); axes2[3,0].legend(fontsize=7)
axes2[3,0].set_title('S4: Ground Effect Factor  (start z=2cm)'); axes2[3,0].set_ylabel('k_ge')
trio(axes2[3,1], s4_v2['t'], s4_v2['p'][:,2]*100, s4_v11['p'][:,2]*100, s4_pb['p'][:,2]*100,
     'S4: Altitude (cm)  -- deepest GE zone', 'z (cm)')
trio(axes2[3,2], s4_v2['t'], s4_v2['v'][:,2], s4_v11['v'][:,2], s4_pb['v'][:,2],
     'S4: Vertical Velocity', 'vz (m/s)')

for ax in axes2.flat: ax.set_xlabel('t (s)', fontsize=8); ax.grid(alpha=0.3)
fig2.tight_layout()
fig2.savefig('extreme_cases.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: traj_comparison.png  +  extreme_cases.png')

In [ ]:
# ── Cell 7: Summary Report ────────────────────────────────────────────────────
print('=' * 65)
print('   FINAL TESTING SUMMARY  --  Swift v2.0 vs v1.1 vs PyBullet')
print('=' * 65)

print('\n[Standard 6s Trajectory with Wind]')
print(f'  Swift v2.0 vs PyBullet : max dev = {dev_v2.max():.2f} mm')
print(f'  Swift v1.1 vs PyBullet : max dev = {dev_v11.max():.2f} mm')
imp = (dev_v11.max() - dev_v2.max()) / dev_v11.max() * 100
print(f'  Improvement from v1.1->v2.0 : {imp:+.1f}% vs PyBullet')

print('\n[S1] Near-ground hover (z=5 cm) -- Ground Effect')
print(f'  Max k_ge v2.0          : {s1_v2["ge"].max():.4f}')
print(f'  Swift v2.0 vs PyBullet : {np.linalg.norm(s1_v2["p"]-s1_pb["p"],axis=1).max()*1000:.2f} mm max')
print(f'  Swift v1.1 vs PyBullet : {np.linalg.norm(s1_v11["p"]-s1_pb["p"],axis=1).max()*1000:.2f} mm max')

print('\n[S2] High-speed (8 m/s) -- Hub Drag + Blade Flapping')
print(f'  vx diff v2.0 vs v1.1 at t=3s : {abs(s2_v2["v"][-1,0]-s2_v11["v"][-1,0]):.3f} m/s  (hub drag)')
print(f'  Swift v2.0 vs PyBullet        : {np.linalg.norm(s2_v2["p"]-s2_pb["p"],axis=1).max()*1000:.2f} mm max')
print(f'  Swift v1.1 vs PyBullet        : {np.linalg.norm(s2_v11["p"]-s2_pb["p"],axis=1).max()*1000:.2f} mm max')

print('\n[S3] Extreme wind (10 m/s crosswind)')
print(f'  X drift diff v2.0 vs v1.1 : {abs(s3_v2["p"][-1,0]-s3_v11["p"][-1,0]):.3f} m')
print(f'  Swift v2.0 vs PyBullet    : {np.linalg.norm(s3_v2["p"]-s3_pb["p"],axis=1).max()*1000:.2f} mm max')
print(f'  Swift v1.1 vs PyBullet    : {np.linalg.norm(s3_v11["p"]-s3_pb["p"],axis=1).max()*1000:.2f} mm max')

print('\n[S4] Very close ground (z=2 cm) -- Deepest Ground Effect')
print(f'  Max k_ge v2.0             : {s4_v2["ge"].max():.4f}')
print(f'  Swift v2.0 vs PyBullet    : {np.linalg.norm(s4_v2["p"]-s4_pb["p"],axis=1).max()*1000:.2f} mm max')

print('\n[Files]')
print(f'  wind  : {wind.__version__}')
print(f'  rlvec : {rlvec.__version__}')
print(f'  pybullet : {pb.__version__}')
print('=' * 65)